<a href="https://colab.research.google.com/github/dee431/-Predictive-Forecasting-of-Care-Load-Placement-Demand-2026-./blob/main/%22Predictive_Forecasting_of_Care_Load_%26_Placement_Demand_2026%22_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dataset**

In [27]:
import numpy as np
import pandas as pd

# Define date range matching the UAC dataset timeline
dates = pd.date_range(start="2021-01-01", end="2025-12-21", freq="D")
np.random.seed(42)

# Generate time-series volume metrics
apprehended = np.random.poisson(lam=15, size=len(dates))
in_cbp = np.random.poisson(lam=45, size=len(dates))
transferred = np.random.poisson(lam=12, size=len(dates))
discharged = np.random.poisson(lam=14, size=len(dates))

# Simulate dynamic HHS Care Load (Cumulative inventory balance)
in_hhs = [2500]
for i in range(1, len(dates)):
    net_change = transferred[i] - discharged[i] + np.random.randint(-3, 4)
    in_hhs.append(max(500, in_hhs[-1] + net_change))

# Format DataFrame to match exact schema
df_gen = pd.DataFrame({
    'Date': dates.strftime('%B %d, %Y'),
    'Children apprehended and placed in CBP custody*': apprehended,
    'Children in CBP custody': in_cbp,
    'Children transferred out of CBP custody': transferred,
    'Children in HHS Care': [f'{val:,}' for val in in_hhs],
    'Children discharged from HHS Care': discharged,
})

# Save to CSV expected by subsequent notebook cells and app.py
df_gen = df_gen.iloc[::-1].reset_index(drop=True)
df_gen.to_csv('HHS_Unaccompanied_Alien_Children_Program.csv', index=False)

print("✅ Dataset 'HHS_Unaccompanied_Alien_Children_Program.csv' successfully generated.")
print(f"Total Rows: {len(df_gen)}")
print(df_gen.head(3))

✅ Dataset 'HHS_Unaccompanied_Alien_Children_Program.csv' successfully generated.
Total Rows: 1816
                Date  Children apprehended and placed in CBP custody*  \
0  December 21, 2025                                               23   
1  December 20, 2025                                               10   
2  December 19, 2025                                               13   

   Children in CBP custody  Children transferred out of CBP custody  \
0                       47                                        6   
1                       38                                        8   
2                       43                                       11   

  Children in HHS Care  Children discharged from HHS Care  
0                  500                                 13  
1                  500                                 23  
2                  507                                  6  


# **%writefile app.py**

In [25]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

st.set_page_config(
    page_title="HHS Care Load & Placement Forecast",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS Injection
st.html("""
    <style>
        .metric-card { background-color: #f8f9fa; padding: 15px; border-radius: 8px; border-left: 5px solid #1f77b4; }
        div[data-testid="stMetricValue"] { font-size: 24px; }
    </style>
""")

st.title("🛡️ HHS UAC Program: Predictive Care Load & Placement Demand")
st.caption("Forward-looking operational intelligence for capacity stress and discharge placement demand.")

@st.cache_data
def load_and_preprocess(file_path):
    df = pd.read_csv(file_path)
    df.columns = ['Date', 'Apprehended_CBP', 'In_CBP_Custody', 'Transferred_Out_CBP', 'In_HHS_Care', 'Discharged_HHS']
    for col in ['Apprehended_CBP', 'In_CBP_Custody', 'Transferred_Out_CBP', 'In_HHS_Care', 'Discharged_HHS']:
        df[col] = df[col].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)')[0].astype(float)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True).set_index('Date').asfreq('D')
    df = df.interpolate(method='time').ffill().bfill()

    df['Net_Flow'] = df['Transferred_Out_CBP'] - df['Discharged_HHS']
    for lag in [1, 7, 14]:
        df[f'HHS_Care_Lag_{lag}'] = df['In_HHS_Care'].shift(lag)
        df[f'Discharged_Lag_{lag}'] = df['Discharged_HHS'].shift(lag)
    for window in [7, 14]:
        df[f'HHS_Care_Roll_Mean_{window}'] = df['In_HHS_Care'].rolling(window).mean()
        df[f'HHS_Care_Roll_Std_{window}'] = df['In_HHS_Care'].rolling(window).std()
    df['DayOfWeek'] = df.index.dayofweek
    df['Month'] = df.index.month
    return df.dropna()

st.sidebar.header("🕹️ Control Panel")
uploaded_file = st.sidebar.file_uploader("Upload UAC CSV Dataset", type=['csv'])

if uploaded_file is not None:
    data = load_and_preprocess(uploaded_file)
else:
    try:
        data = load_and_preprocess('HHS_Unaccompanied_Alien_Children_Program.csv')
    except Exception:
        st.error("Please upload the dataset to continue.")
        st.stop()

model_choice = st.sidebar.selectbox("Select Forecasting Model", ["Gradient Boosting Regressor", "Random Forest Regressor", "Holt-Winters Exponential Smoothing"])
horizon = st.sidebar.slider("Forecast Horizon (Days)", min_value=7, max_value=60, value=14)
capacity_threshold = st.sidebar.number_input("HHS Shelter Capacity Threshold", value=12000, step=500)

train_size = int(len(data) * 0.8)
train, test = data.iloc[:train_size], data.iloc[train_size:]
X_cols = [c for c in data.columns if c not in ['In_HHS_Care', 'Discharged_HHS']]

if model_choice == "Gradient Boosting Regressor":
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
    model.fit(train[X_cols], train['In_HHS_Care'])
    preds = model.predict(test[X_cols])
elif model_choice == "Random Forest Regressor":
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(train[X_cols], train['In_HHS_Care'])
    preds = model.predict(test[X_cols])
else:
    model = ExponentialSmoothing(train['In_HHS_Care'], trend='add', seasonal=None).fit()
    preds = model.forecast(len(test))

test_preds = pd.Series(preds, index=test.index)

col1, col2, col3, col4 = st.columns(4)
current_care = int(data['In_HHS_Care'].iloc[-1])
forecast_care = int(test_preds.iloc[-1])
mape = np.mean(np.abs((test['In_HHS_Care'] - test_preds) / test['In_HHS_Care'])) * 100
breach_risk = (test_preds.iloc[-horizon:] > capacity_threshold).mean() * 100

col1.metric("Current Care Load", f"{current_care:,}")
col2.metric(f"Forecast ({horizon}D Target)", f"{forecast_care:,}", delta=f"{forecast_care - current_care:,}")
col3.metric("Forecast Accuracy", f"{100 - mape:.1f}%")
col4.metric("Capacity Breach Probability", f"{breach_risk:.1f}%")

st.markdown("---")

st.subheader("📈 Care Load Forecast & Confidence Interval")
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index[-60:], y=train['In_HHS_Care'].iloc[-60:], mode='lines', name='Historical Load', line=dict(color='#2c3e50', width=2)))
fig.add_trace(go.Scatter(x=test.index, y=test['In_HHS_Care'], mode='lines', name='Actual Load', line=dict(color='#7f8c8d', width=1.5)))
fig.add_trace(go.Scatter(x=test.index, y=test_preds, mode='lines', name=f'Forecast ({model_choice})', line=dict(color='#e74c3c', width=2.5)))

std_err = np.std(test['In_HHS_Care'] - test_preds)
upper_bound = test_preds + 1.96 * std_err
lower_bound = test_preds - 1.96 * std_err

fig.add_trace(go.Scatter(x=test.index.tolist() + test.index.tolist()[::-1],
                         y=upper_bound.tolist() + lower_bound.tolist()[::-1],
                         fill='toself', fillcolor='rgba(231, 76, 60, 0.2)',
                         line=dict(color='rgba(255,255,255,0)'), name='95% Confidence Interval'))

fig.add_hline(y=capacity_threshold, line_dash="dash", line_color="black", annotation_text="Max Capacity Threshold")
fig.update_layout(xaxis_title="Date", yaxis_title="Children in HHS Care", hovermode="x unified", height=450)
st.plotly_chart(fig, use_container_width=True)

col_a, col_b = st.columns(2)
with col_a:
    st.subheader("🚪 Estimated Discharge Demand")
    fig_disc = go.Figure()
    fig_disc.add_trace(go.Bar(x=test.index[-30:], y=test['Discharged_HHS'].iloc[-30:], name="Actual Discharges", marker_color='#2ecc71'))
    fig_disc.add_trace(go.Scatter(x=test.index[-30:], y=test['Net_Flow'].iloc[-30:], name="Net Flow Pressure", line=dict(color='#e67e22', width=2)))
    fig_disc.update_layout(xaxis_title="Date", yaxis_title="Count", height=350)
    st.plotly_chart(fig_disc, use_container_width=True)

with col_b:
    st.subheader("🚨 Early-Warning Indicators")
    recent_net = data['Net_Flow'].iloc[-7:].mean()
    if recent_net > 0:
        st.warning(f"⚠️ **Inflow Exceeds Outflow**: Net intake is averaging +{recent_net:.1f} children/day over the last 7 days.")
    else:
        st.success(f"✅ **Outflow Outpaces Inflow**: Net discharge is averaging {recent_net:.1f} children/day over the last 7 days.")

    st.write(f"• **Surge Lead Time**: ~{int(horizon/2)} days based on current moving pressure.")
    st.write(f"• **Model Robustness Index**: Stable (MAE: {np.mean(np.abs(test['In_HHS_Care'] - test_preds)):.2f})")

Overwriting app.py


# **Streamlit Server**

In [31]:
import urllib

# Install localtunnel globally
!npm install -g localtunnel -q

# Run Streamlit with CORS and XSRF protection disabled for localtunnel tunneling
!streamlit run app.py --server.enableCORS=false --server.enableXsrfProtection=false &> /dev/null &

# Output connection credentials
print("1. Copy this Password IP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("2. Click the link below and enter the Password IP to launch your dashboard:\n")

!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼1. Copy this Password IP: 34.24.3.95
2. Click the link below and enter the Password IP to launch your dashboard:

⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://major-donkeys-reply.loca.lt
^C


In [33]:
# 1. Install Dependencies, Cloudflare Tunnel CLI, & Generate Dataset File
!pip install -q streamlit plotly pandas numpy scikit-learn statsmodels
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb

import numpy as np
import pandas as pd

dates = pd.date_range(start="2021-01-01", end="2025-12-21", freq="D")
np.random.seed(42)
apprehended = np.random.poisson(lam=15, size=len(dates))
in_cbp = np.random.poisson(lam=45, size=len(dates))
transferred = np.random.poisson(lam=12, size=len(dates))
discharged = np.random.poisson(lam=14, size=len(dates))

in_hhs = [2500]
for i in range(1, len(dates)):
    net_change = transferred[i] - discharged[i] + np.random.randint(-3, 4)
    in_hhs.append(max(500, in_hhs[-1] + net_change))

df_gen = pd.DataFrame({
    'Date': dates.strftime('%B %d, %Y'),
    'Children apprehended and placed in CBP custody*': apprehended,
    'Children in CBP custody': in_cbp,
    'Children transferred out of CBP custody': transferred,
    'Children in HHS Care': [f'{val:,}' for val in in_hhs],
    'Children discharged from HHS Care': discharged,
}).iloc[::-1].reset_index(drop=True)

df_gen.to_csv('HHS_Unaccompanied_Alien_Children_Program.csv', index=False)
print("✅ Setup completed and HHS_Unaccompanied_Alien_Children_Program.csv created successfully.")

(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.9.1) over (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
✅ Setup completed and HHS_Unaccompanied_Alien_Children_Program.csv created successfully.


In [35]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

st.set_page_config(
    page_title="HHS Care Load & Placement Forecast",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS Injection
st.html("""
    <style>
        .metric-card { background-color: #f8f9fa; padding: 15px; border-radius: 8px; border-left: 5px solid #1f77b4; }
        div[data-testid="stMetricValue"] { font-size: 24px; }
    </style>
""")

st.title("🛡️ HHS UAC Program: Predictive Care Load & Placement Demand")
st.caption("Forward-looking operational intelligence for capacity stress and discharge placement demand.")

@st.cache_data
def load_and_preprocess(file_path):
    df = pd.read_csv(file_path)
    df.columns = ['Date', 'Apprehended_CBP', 'In_CBP_Custody', 'Transferred_Out_CBP', 'In_HHS_Care', 'Discharged_HHS']
    for col in ['Apprehended_CBP', 'In_CBP_Custody', 'Transferred_Out_CBP', 'In_HHS_Care', 'Discharged_HHS']:
        df[col] = df[col].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)')[0].astype(float)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True).set_index('Date').asfreq('D')
    df = df.interpolate(method='time').ffill().bfill()

    df['Net_Flow'] = df['Transferred_Out_CBP'] - df['Discharged_HHS']
    for lag in [1, 7, 14]:
        df[f'HHS_Care_Lag_{lag}'] = df['In_HHS_Care'].shift(lag)
        df[f'Discharged_Lag_{lag}'] = df['Discharged_HHS'].shift(lag)
    for window in [7, 14]:
        df[f'HHS_Care_Roll_Mean_{window}'] = df['In_HHS_Care'].rolling(window).mean()
        df[f'HHS_Care_Roll_Std_{window}'] = df['In_HHS_Care'].rolling(window).std()
    df['DayOfWeek'] = df.index.dayofweek
    df['Month'] = df.index.month
    return df.dropna()

st.sidebar.header("🕹️ Control Panel")
uploaded_file = st.sidebar.file_uploader("Upload UAC CSV Dataset", type=['csv'])

if uploaded_file is not None:
    data = load_and_preprocess(uploaded_file)
else:
    try:
        data = load_and_preprocess('HHS_Unaccompanied_Alien_Children_Program.csv')
    except Exception:
        st.error("Please upload the dataset to continue.")
        st.stop()

model_choice = st.sidebar.selectbox("Select Forecasting Model", ["Gradient Boosting Regressor", "Random Forest Regressor", "Holt-Winters Exponential Smoothing"])
horizon = st.sidebar.slider("Forecast Horizon (Days)", min_value=7, max_value=60, value=14)
capacity_threshold = st.sidebar.number_input("HHS Shelter Capacity Threshold", value=12000, step=500)

train_size = int(len(data) * 0.8)
train, test = data.iloc[:train_size], data.iloc[train_size:]
X_cols = [c for c in data.columns if c not in ['In_HHS_Care', 'Discharged_HHS']]

if model_choice == "Gradient Boosting Regressor":
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
    model.fit(train[X_cols], train['In_HHS_Care'])
    preds = model.predict(test[X_cols])
elif model_choice == "Random Forest Regressor":
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(train[X_cols], train['In_HHS_Care'])
    preds = model.predict(test[X_cols])
else:
    model = ExponentialSmoothing(train['In_HHS_Care'], trend='add', seasonal=None).fit()
    preds = model.forecast(len(test))

test_preds = pd.Series(preds, index=test.index)

col1, col2, col3, col4 = st.columns(4)
current_care = int(data['In_HHS_Care'].iloc[-1])
forecast_care = int(test_preds.iloc[-1])
mape = np.mean(np.abs((test['In_HHS_Care'] - test_preds) / test['In_HHS_Care'])) * 100
breach_risk = (test_preds.iloc[-horizon:] > capacity_threshold).mean() * 100

col1.metric("Current Care Load", f"{current_care:,}")
col2.metric(f"Forecast ({horizon}D Target)", f"{forecast_care:,}", delta=f"{forecast_care - current_care:,}")
col3.metric("Forecast Accuracy", f"{100 - mape:.1f}%")
col4.metric("Capacity Breach Probability", f"{breach_risk:.1f}%")

st.markdown("---")

st.subheader("📈 Care Load Forecast & Confidence Interval")
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index[-60:], y=train['In_HHS_Care'].iloc[-60:], mode='lines', name='Historical Load', line=dict(color='#2c3e50', width=2)))
fig.add_trace(go.Scatter(x=test.index, y=test['In_HHS_Care'], mode='lines', name='Actual Load', line=dict(color='#7f8c8d', width=1.5)))
fig.add_trace(go.Scatter(x=test.index, y=test_preds, mode='lines', name=f'Forecast ({model_choice})', line=dict(color='#e74c3c', width=2.5)))

std_err = np.std(test['In_HHS_Care'] - test_preds)
upper_bound = test_preds + 1.96 * std_err
lower_bound = test_preds - 1.96 * std_err

fig.add_trace(go.Scatter(x=test.index.tolist() + test.index.tolist()[::-1],
                         y=upper_bound.tolist() + lower_bound.tolist()[::-1],
                         fill='toself', fillcolor='rgba(231, 76, 60, 0.2)',
                         line=dict(color='rgba(255,255,255,0)'), name='95% Confidence Interval'))

fig.add_hline(y=capacity_threshold, line_dash="dash", line_color="black", annotation_text="Max Capacity Threshold")
fig.update_layout(xaxis_title="Date", yaxis_title="Children in HHS Care", hovermode="x unified", height=450)
st.plotly_chart(fig, use_container_width=True)

col_a, col_b = st.columns(2)
with col_a:
    st.subheader("🚪 Estimated Discharge Demand")
    fig_disc = go.Figure()
    fig_disc.add_trace(go.Bar(x=test.index[-30:], y=test['Discharged_HHS'].iloc[-30:], name="Actual Discharges", marker_color='#2ecc71'))
    fig_disc.add_trace(go.Scatter(x=test.index[-30:], y=test['Net_Flow'].iloc[-30:], name="Net Flow Pressure", line=dict(color='#e67e22', width=2)))
    fig_disc.update_layout(xaxis_title="Date", yaxis_title="Count", height=350)
    st.plotly_chart(fig_disc, use_container_width=True)

with col_b:
    st.subheader("🚨 Early-Warning Indicators")
    recent_net = data['Net_Flow'].iloc[-7:].mean()
    if recent_net > 0:
        st.warning(f"☑☑ **Inflow Exceeds Outflow**: Net intake is averaging +{recent_net:.1f} children/day over the last 7 days.")
    else:
        st.success(f"✅ **Outflow Outpaces Inflow**: Net discharge is averaging {recent_net:.1f} children/day over the last 7 days.")

    st.write(f"• **Surge Lead Time**: ~{int(horizon/2)} days based on current moving pressure.")
    st.write(f"• **Model Robustness Index**: Stable (MAE: {np.mean(np.abs(test['In_HHS_Care'] - test_preds)):.2f})")

Overwriting app.py


In [36]:
# 3. Start Background Streamlit Server & Expose via Cloudflare Tunnel
import subprocess
import time

# Terminate any duplicate running streamlit instances
subprocess.run("pkill streamlit", shell=True)

# Start Streamlit
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"])
time.sleep(5)

print("🌐 Streamlit server online. Launching secure public tunnel...\n")
!cloudflared tunnel --url http://localhost:8501

🌐 Streamlit server online. Launching secure public tunnel...

2026-09-18T18:12:17Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-18T18:12:17Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-18T18:12:21Z INF +--------------------------------------------------------------------------------------------+
2026-09-18T18:12:21Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-18T18:12:21